# 第3回(6/16)の宿題： 手書きの数字を20回程度様々な数字を書いてみて，分類精度を確かめて，テストデータの精度と比較しなさい（次回，入力フォームを用意します）．



## 前回の宿題2 →第3回実習続き. Scikit-Learnの他のモデル，ハイパーパラメータを試し，全データを作った訓練で，どこまで精度高められるか試す．ただし，実習中はデータ数1000でやってみる．

例えば，
### a1. 線形ロジスティック回帰の場合は，正則化 elastic-net を使って，L1_ratio や, alpha を変更

### a2. SVM の場合は，カーネルの種類，C や gamma 等のパラメータ値

### a3. 多層パーセプトロンの場合は隠れ層の数と，ニューロン数の変更


等．
### b. 下記サイトを参考に他のアルゴリズム（授業で用いなかったものも可）を試し精度向上を検討しなさい．

https://scikit-learn.org/stable/supervised_learning.html  






# 分類 (classification)
### MNIST data: hand-written characters of digits
### 手書き数字認識
NIST: National Institute of Standards and Technology, US
アメリカ国立標準技術研究所

MNIST: Modified National Institute of Standards and Technology database

## 1. Shallow (浅い) 機械学習を使う

## 2. Perceptron (単層，多層 ← 深層じゃないもの) を使う

## 3. Deep Learning (深層学習) を使う



In [ ]:
from sklearn import datasets, metrics
from sklearn.datasets import fetch_openml
import numpy as np
import matplotlib.pyplot as plt
 # データはインターネットから取ってきて，
 # 訓練(学習)用の入力(x)・出力(y)，テスト用の入力(x)・出力(y) に代入
# (x_train, y_train), (x_test, y_test) = mnist.load_data()
x, y = fetch_openml('mnist_784', version=1, return_X_y=True)

In [ ]:
x = x.values         # オブジェクト x に，x.values とすると，values 属性(プロパティ)を取り出せる (numpy の ndarray 配列形式になる)
y = y.astype(int).values    # ラベルデータとして整数(int)型にしたいので，astype(int)で変換（しないと 1.00 のような浮動小数点型 (float) になる

## データを訓練 (train) とテスト (test) に分ける
```python
x_train, x_test = x[0:60000], x[60000:70000]
y_train, y_test = y[0:60000], y[60000:70000]
```
[a:b] と書くのは，配列 ndarray の a 個目(「0」が1個目）から，b - 1 個目までを取り出す操作．最初と最後は，省略していいので，
```python
x_train, x_test = x[:60000], x[60000:]
y_train, y_test = y[:60000], y[60000:]
```
と書いても一緒．

ハイパーパラメータチューニングやモデル比較をする際は，更に，訓練データから検証データ (validation (val) ) を分けておく必要があるが，今日は valid は使わない（教科書 p.326）．

In [ ]:
x_train, x_test = x[0:60000], x[60000:70000]
y_train, y_test = y[0:60000], y[60000:70000]

### 計算時間短縮のため，データを60000個のうち最初の1000個だけを使う
Number of data is restricted to 1000 because of the computation cost
```python
data_size = 1000
x_train1 = x_train[0:data_size]
y_train1 = y_train[0:data_size]
x_test1 = x_test[0:data_size]
y_test1 = y_test[0:data_size]
```

In [ ]:
data_size = 1000
x_train1 = x_train[0:data_size]
y_train1 = y_train[0:data_size]
x_test1 = x_test[0:data_size]
y_test1 = y_test[0:data_size]

# ここから，教科書3章に出てくる様々な分類器を使って，MNIST 分類（文字認識（パターン認識），予測）をする．

## 0. ロジスティック回帰 (Logistic regression)


In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
clf_lf = LogisticRegression()
clf_lf.fit(x_train1, y_train1)
print( "訓練データの正答率" )
print( clf_lf.score(x_train1, y_train1) )
print( "テストデータの正答率" )
print( clf_lf.score(x_test1, y_test1) )
predicted = clf_lf.predict(x_test1)
expected = y_test1
print(metrics.classification_report(expected, predicted))
print(metrics.confusion_matrix(expected, predicted))

### 過学習，および過学習回避のための正則化 (regularization)

どうも過学習をしているように見える．

学習データ数 1000 に対して，特徴量＝説明変数＝各画素値が 784 と多いのかも知れない（実際，特徴量＞データ数だと，100%を達成できる）．

重みの学習にペナルティをかけることで，訓練データの精度を下げて（過学習を抑制），テストデータの精度上げて，将来の未知のデータに対する精度を高めることを試みる．

線形回帰での方法は，2つあり，
- **L2正則化=リッジ回帰** : あまり大きな重みを取らせない．どれくらいの強さでペナルティをかけるかで，C というパラメータを使う.C が大きいと，訓練データを正確に学習しようとする（過学習（チートみたいなもの）の可能性あり）．C が小さいと，訓練データを全ては正確には識別しようとはしない（無理に精度があげるために重みをあり得ない値にしようとすると，チートと判定して，ペナルティ（罰）を与えてスコアを落とす）．
- **L1正則化=LASSO** : 重みを取る変数の数を厳選する（=疎（スパース=sparse）にする）．精度向上に関係無さそうな画素の重みは積極的に 0 にする．
- **エラスティックネット (elastic net)** : 上記の両方をやる．ハイパーパラメータは2個で，C と，l1_ratio (L1正則化項（罰則項）のL2に対する比率) を決める．

In [ ]:
clf_lf = LogisticRegression(C=1)
clf_lf.fit(x_train1, y_train1)
print( "訓練データの正答率" )
print( clf_lf.score(x_train1, y_train1) )
print( "テストデータの正答率" )
print( clf_lf.score(x_test1, y_test1) )
predicted = clf_lf.predict(x_test1)
expected = y_test1
print(metrics.classification_report(expected, predicted))
print(metrics.confusion_matrix(expected, predicted))

# 1. Shallow (浅い) 機械学習

SVM (1.1) と，Random Forest (1.2) を試す

# 1.1. SVM : サポートベクトルマシン
support vector machine

非線形のモデルで，汎化性能が高い．

3.4.2 節 (p.77) の SVC （サポートベクトル分類器）を使う．



In [ ]:
from sklearn.svm import SVC

### 1.1.1. 線形カーネル（＝カーネルトリックを使わない）

In [ ]:
clf_svml = SVC(kernel='linear')
clf_svml.fit(x_train1, y_train1)
print( "訓練データの正答率" )
print( clf_svml.score(x_train1, y_train1) )
print( "テストデータの正答率" )
print( clf_svml.score(x_test1, y_test1) )
predicted = clf_svml.predict(x_test1)
expected = y_test1
print(metrics.classification_report(expected, predicted))
print(metrics.confusion_matrix(expected, predicted))

### 1.1.2. カーネルを変えたサポートベクトルマシン

RBF カーネル（3.5.2節 p.80-81，ガウスカーネルとも呼ぶ）や，多項式カーネル（2次関数，3次関数，etc. scikit-learn のマニュアル https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC 等も参考に）を使ったモデルを使う．

In [ ]:
clf_svmk = SVC(kernel='rbf', C=10, gamma="scale")
clf_svmk.fit(x_train1, y_train1)
print( "訓練データの正答率" )
print( clf_svmk.score(x_train1, y_train1) )
print( "テストデータの正答率" )
print( clf_svmk.score(x_test1, y_test1) )
predicted = clf_svmk.predict(x_test1)
expected = y_test1
print(metrics.classification_report(expected, predicted))
print(metrics.confusion_matrix(expected, predicted))


# 1.2. RF: Randome Forest
ランダムフォレスト（ランダムな特徴に基づく決定木が数多くあり，それらの多数決(アンサンブル学習）をする．


In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
clf_rf = RandomForestClassifier(n_estimators=100,      # 決定木の数
                                    max_depth=20,         # 木の最大深さ
                                    max_features=200)  # 使う特徴量の最大数
clf_rf.fit(x_train1, y_train1)
print( "訓練データの正答率" )
print( clf_rf.score(x_train1, y_train1) )
print( "テストデータの正答率" )
print( clf_rf.score(x_test1, y_test1) )
predicted = clf_rf.predict(x_test1)
expected = y_test1
print(metrics.classification_report(expected, predicted))
print(metrics.confusion_matrix(expected, predicted))


In [ ]:
import pandas as pd

## 1.3 ハイパーパラメータチューニング

SVM で RBFカーネル (Radial Basis Function=動径基底 核関数) では，C (ソフトマージン=訓練時にどれくらい間違えを許さないか(大きいほど間違えに厳しい=ペナルティが大きい))， γ (RBF=ガウス関数の広がり．小さいほど細かく見る) を変えると精度が大きく変わるので，これらハイパーパラメータを適切に決める必要がある．

そこで，ハイパーパラメータ探索をするが，最も簡易なやり方はグリッドサーチ (GridSearch) で，定められた値の組み合わせを全部試す．

In [ ]:
from sklearn.model_selection import GridSearchCV
params = {"gamma":[1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7],
              "C": [1, 10, 100, 1e3, 1e4, 1e5]}
gs = GridSearchCV(estimator=clf_svmk, param_grid=params, scoring="accuracy", cv=2)
gs.fit(x_train1, y_train1)
print(gs.best_params_)
print(gs.best_score_)
results = pd.DataFrame(gs.cv_results_)
results[["param_C", "param_gamma"]] = results[["param_C", "param_gamma"]].astype(np.float64)
scores_matrix = results.pivot(
            index="param_gamma", columns="param_C", values="mean_test_score"
        )


### ハイパーパラメータと精度の関係をヒートマップで表示する

よく論文や報告書（レポート）で見る形式

In [ ]:
fig, axes = plt.subplots(ncols=1, sharey=True)
ax1 = axes
im = ax1.imshow(scores_matrix)
gammas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7]
Cs = [1, 10, 100, 1e3, 1e4, 1e5]
ax1.set_xticks(np.arange(len(Cs)))
ax1.set_xticklabels(["{:.0E}".format(x) for x in Cs])
ax1.set_xlabel("C", fontsize=15)
ax1.set_yticks(np.arange(len(gammas)))
ax1.set_yticklabels(["{:.0E}".format(x) for x in gammas])
ax1.set_ylabel("gamma", fontsize=15)

    # Rotate the tick labels and set their alignment.
plt.setp(ax1.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
fig.subplots_adjust(right=0.8)
cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
fig.colorbar(im, cax=cbar_ax)
cbar_ax.set_ylabel("mean_test_score", rotation=-90, va="bottom", fontsize=15)


# 2. パーセプトロン，Perceptron
- 最も簡素なニューラルネットワーク (Most simple neural network)
- マカロックピッツの形式ニューロン[1943]，ロゼンブラットの学習アルゴリズム[1958]
- ローゼンブラットの単層パーセプトロン
- 1960年代に線形モデルと等価であることが示された
- Mucolloch-Pitts, formal neuron model [1943], and the learning by Rosenblatt [1958].
- Rosenblatt's single layer perceptron
- In 1960s, equivalence to linear model is suggested.

In [ ]:
from sklearn.linear_model import Perceptron

In [ ]:
clf_p = Perceptron()
clf_p.fit(x_train1, y_train1)
print( clf_p.score(x_train1, y_train1) )
print( clf_p.score(x_test1, y_test1) )
predicted = clf_p.predict(x_test1)
expected = y_test1
print(metrics.classification_report(expected, predicted))
print(metrics.confusion_matrix(expected, predicted))


## 5. 多層パーセプトロン：MLP: multi-layered perceptron
- ラメルハート，ヒントン，ウイリアムズの3人が，1986年 Nature で発表．Rumelhart, Hinton, Williams (Nature, 1986)
- シグモイド関数（ロジスティック関数）を使い，多層を実現
- $y = \dfrac{1}{1+\exp(-x)}$

In [ ]:
from sklearn.neural_network import MLPClassifier

In [ ]:
clf_mlp1 = MLPClassifier(hidden_layer_sizes=(),activation='identity')
clf_mlp1.fit(x_train1, y_train1)
print( clf_mlp1.score(x_train1, y_train1) )
print( clf_mlp1.score(x_test1, y_test1) )
predicted = clf_mlp1.predict(x_test1)
expected = y_test1
print(metrics.classification_report(expected, predicted))
print(metrics.confusion_matrix(expected, predicted))


In [ ]:
clf_mlp2 = MLPClassifier(hidden_layer_sizes=(1),activation='identity')
clf_mlp2.fit(x_train1, y_train1)
print( clf_mlp2.score(x_train1, y_train1) )
print( clf_mlp2.score(x_test1, y_test1) )
predicted = clf_mlp2.predict(x_test1)
expected = y_test1
print(metrics.classification_report(expected, predicted))
print(metrics.confusion_matrix(expected, predicted))


In [ ]:
clf_mlp3 = MLPClassifier(hidden_layer_sizes=(2,2),activation='identity')
clf_mlp3.fit(x_train1, y_train1)
print( clf_mlp3.score(x_train1, y_train1) )
print( clf_mlp3.score(x_test1, y_test1) )
predicted = clf_mlp3.predict(x_test1)
expected = y_test1
print(metrics.classification_report(expected, predicted))
print(metrics.confusion_matrix(expected, predicted))



# **実際に手書き数字を認識させる**

### 手書き文字の準備

In [ ]:
from IPython.display import HTML, Image
from google.colab.output import eval_js
from base64 import b64decode
from io import BytesIO
from PIL import Image

def centerOfMass(img):
  """
  画像imgの重心位置を返す
  :param PIL.Image.Image img: グレイスケール画像
  :rtype:  (int, int)
  :return: (重心のX座標, 重心のY座標)
  """
  import numpy as np

  m = np.asarray(img)	# NumPy配列に変換
  ht, wd = m.shape
  sum = np.sum(m)

  # https://stackoverflow.com/questions/37519238
  dx = np.sum(m, axis=0)	# 各列の合計からなるベクトル
  dy = np.sum(m, axis=1)	# 各行の合計からなるベクトル

  # np.arange(wd) == [0, 1, 2, ..., wd-1]
  cx = np.sum(dx * np.arange(wd)) / sum
  cy = np.sum(dy * np.arange(ht)) / sum

  return (int(np.rint(cx)), int(np.rint(cy)))

## 関数 drawdigit() の定義：手書き
## 時間がなければいったん 3. へスキップしてディープラーニングをやる

JavaScript Canvas 機能を使って書いた文字を配列 ndarray に入れる．

関数の返り値 (return) は，28×28 の画像をさらに，NumPy の ndarray にし，1×784 の1次元配列に変換（Flatten）したものを返す．  
これで，そのまま，clf.predict に渡して認識（予測）できる．

In [ ]:
def drawdigit():
  canvas_html = """
  <canvas width=%d height=%d></canvas>
  <button>数字保存！</button>
  <script>
  var canvas = document.querySelector('canvas')
  var ctx = canvas.getContext('2d')
  canvas.style.border = "2px solid"
  ctx.lineWidth = %d
  var button = document.querySelector('button')
  var mouse = {x: 0, y: 0}
  canvas.addEventListener('mousemove', function(e) {
    mouse.x = e.pageX - this.offsetLeft
    mouse.y = e.pageY - this.offsetTop
  })
  canvas.onmousedown = ()=>{
    ctx.beginPath()
    ctx.moveTo(mouse.x, mouse.y)
    canvas.addEventListener('mousemove', onPaint)
  }
  canvas.onmouseup = ()=>{
    canvas.removeEventListener('mousemove', onPaint)
  }
  var onPaint = ()=>{
    ctx.lineTo(mouse.x, mouse.y)
    ctx.stroke()
  }
  var data = new Promise(resolve=>{
    button.onclick = ()=>{
      resolve(canvas.toDataURL('image/png'))
    }
  })
  </script>
  """

  display(HTML(canvas_html % (200, 200, 8)))
  data = eval_js("data")
  binary = b64decode(data.split(',')[1])
  img = Image.open(BytesIO(binary))

  background = Image.new("RGB", img.size, (255, 255, 255))
  background.paste(img, img)
  img = background

  img = img.convert('L')	# grayscale
  img = ImageOps.invert(img)	# negate
  img = img.crop(box=img.getbbox())	# crop
  wd, ht = img.size
  # アスペクト比を変えずに20x20の矩形に収める
  if wd < ht:
    wd = wd * 20 // ht
    ht = 20
  else:
    ht = ht * 20 // wd
    wd = 20
  #img = img.resize((wd, ht), Image.Resampling.LANCZOS)
  img = img.resize((wd, ht))
  cx, cy = centerOfMass(img)	# 重心

  # 重心を中心にして28x28の矩形の中に配置
  bgImg = Image.new(img.mode, (28, 28), color=0)
  ox = -cx + 28 // 2
  oy = -cy + 28 // 2
  bgImg.paste(img, (ox, oy))
  img = bgImg
  display(img)
  return np.asarray(img).astype(np.float32).reshape(1, 784)


## drawdigit() を実行し数字を手書きする．
```python
suuji = drawdigit()
```

In [ ]:
suuji = drawdigit()

## clf_***.predict に suuji を入れて認識させる
```python
print( clf_***.predict(suuji) )
```
または
```python
print( clf_***.predict(suuji)[0] )
```
[0] は配列の1つ目の値だけを取る操作

In [ ]:
print( clf_lf.predict(suuji)[0] )

# 3. Deep Learning を使う

Pytorch を使ったディープラーニングをやってみる．

----
「Pytorch を使って，単層パーセプトロンによる MNIST 分類モデルを作る．データは，上で使った x_train1, y_train1, x_test1, y_test1で，Epoch 毎の Loss, Accuracy の表示と，最後に Loss, Accuracy の曲線をみたい．fit 中はプログレスバーを表示して欲しいのと．モデルの構造も表示して欲しい．」

----


## PyTorchによるニューラルネットワーク

### 単層パーセプトロンモデル

ここでは、PyTorchを使用して単層パーセプトロン（MLPの一種で、隠れ層が一つ）を構築し、MNISTデータの分類を行います。訓練中のエポックごとの損失と精度を追跡し、学習の進行を視覚化します。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# データをPyTorchのテンソルに変換
# 単層パーセプトロンはフラットな入力を期待するので、画像を1次元にリシェイプします。
# xデータはfloat32、yデータはLong（分類タスクのターゲット）
X_train_tensor_mlp = torch.tensor(x_train1, dtype=torch.float32).view(-1, 784)
y_train_tensor_mlp = torch.tensor(y_train1, dtype=torch.long)
X_test_tensor_mlp = torch.tensor(x_test1, dtype=torch.float32).view(-1, 784)
y_test_tensor_mlp = torch.tensor(y_test1, dtype=torch.long)

# 画像データを[0, 1]の範囲に正規化 (MNISTデータは通常0-255)
X_train_tensor_mlp /= 255.0
X_test_tensor_mlp /= 255.0

# DatasetとDataLoaderの作成
train_dataset_mlp = TensorDataset(X_train_tensor_mlp, y_train_tensor_mlp)
test_dataset_mlp = TensorDataset(X_test_tensor_mlp, y_test_tensor_mlp)

batch_size = 64
train_loader_mlp = DataLoader(train_dataset_mlp, batch_size=batch_size, shuffle=True)
test_loader_mlp = DataLoader(test_dataset_mlp, batch_size=batch_size, shuffle=False)

# 単層パーセプトロンモデルの定義
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        # 入力層 (784ピクセル) から隠れ層 (例: 128ニューロン)
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU() # 活性化関数
        # 隠れ層から出力層 (10クラス)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# モデルのインスタンス化、損失関数、オプティマイザの定義
model_mlp = SimpleMLP()
criterion_mlp = nn.CrossEntropyLoss()
optimizer_mlp = optim.Adam(model_mlp.parameters(), lr=0.001)

# デバイスの設定 (GPUが利用可能ならGPUを使う)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_mlp.to(device)

# 訓練中のメトリクスを保存するリスト
train_losses_mlp = []
train_accuracies_mlp = []
test_losses_mlp = []
test_accuracies_mlp = []

# モデルの訓練
epochs = 10
print("\n単層パーセプトロンモデル訓練開始...")
for epoch in range(epochs):
    model_mlp.train()  # 訓練モード
    running_loss = 0.0
    correct_train_epoch = 0
    total_train_epoch = 0
    for inputs, labels in train_loader_mlp:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_mlp.zero_grad() # 勾配をゼロクリア
        outputs = model_mlp(inputs)
        loss = criterion_mlp(outputs, labels)
        loss.backward()       # 逆伝播
        optimizer_mlp.step()      # パラメータ更新

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train_epoch += labels.size(0)
        correct_train_epoch += (predicted == labels).sum().item()

    train_loss_epoch = running_loss / len(train_loader_mlp)
    train_accuracy_epoch = 100 * correct_train_epoch / total_train_epoch
    train_losses_mlp.append(train_loss_epoch)
    train_accuracies_mlp.append(train_accuracy_epoch)

    # エポック終了後のテストデータでの評価
    model_mlp.eval() # 評価モード
    correct_test_epoch = 0
    total_test_epoch = 0
    running_test_loss = 0.0
    with torch.no_grad(): # 勾配計算を無効化
        for inputs, labels in test_loader_mlp:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_mlp(inputs)
            loss = criterion_mlp(outputs, labels)
            running_test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_test_epoch += labels.size(0)
            correct_test_epoch += (predicted == labels).sum().item()

    test_loss_epoch = running_test_loss / len(test_loader_mlp)
    test_accuracy_epoch = 100 * correct_test_epoch / total_test_epoch
    test_losses_mlp.append(test_loss_epoch)
    test_accuracies_mlp.append(test_accuracy_epoch)

    print(f"Epoch {epoch+1}, Train Loss: {train_loss_epoch:.4f}, Train Acc: {train_accuracy_epoch:.2f}%, Test Loss: {test_loss_epoch:.4f}, Test Acc: {test_accuracy_epoch:.2f}%")

print("単層パーセプトロンモデル訓練完了！")

# モデルの評価 (最終エポックの結果)
model_mlp.eval() # 評価モード
correct_mlp = 0
total_mlp = 0
all_predicted_mlp = []
all_labels_mlp = []
with torch.no_grad(): # 勾配計算を無効化
    for inputs, labels in test_loader_mlp:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model_mlp(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total_mlp += labels.size(0)
        correct_mlp += (predicted == labels).sum().item()
        all_predicted_mlp.extend(predicted.cpu().numpy())
        all_labels_mlp.extend(labels.cpu().numpy())

accuracy_mlp = 100 * correct_mlp / total_mlp
print(f"\n最終テストデータでの単層パーセプトロンモデル精度: {accuracy_mlp:.2f}%")

# 詳細なレポート
print("\n単層パーセプトロンモデルの分類レポート:")
print(classification_report(all_labels_mlp, all_predicted_mlp))

print("\n単層パーセプトロンモデルの混同行列:")
print(confusion_matrix(all_labels_mlp, all_predicted_mlp))

# 学習曲線 (Learning Curves) のプロット
plt.figure(figsize=(12, 5))

# 損失のプロット
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), train_losses_mlp, label='Train Loss')
plt.plot(range(1, epochs + 1), test_losses_mlp, label='Test Loss')
plt.title('Loss per Epoch (Single-Layer Perceptron)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 精度のプロット
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), train_accuracies_mlp, label='Train Accuracy')
plt.plot(range(1, epochs + 1), test_accuracies_mlp, label='Test Accuracy')
plt.title('Accuracy per Epoch (Single-Layer Perceptron)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 畳み込みニューラルネットワークによる深層学習モデル

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm # tqdm をインポート

# データをPyTorchのテンソルに変換
# CNNの入力形式に合わせて、画像を (batch_size, channels, height, width) にリシェイプします。
# MNISTはグレースケールなので channels=1, 画像サイズは28x28です。
# xデータはfloat32、yデータはLong（分類タスクのターゲット）
X_train_tensor_cnn = torch.tensor(x_train1, dtype=torch.float32).view(-1, 1, 28, 28)
y_train_tensor_cnn = torch.tensor(y_train1, dtype=torch.long)
X_test_tensor_cnn = torch.tensor(x_test1, dtype=torch.float32).view(-1, 1, 28, 28)
y_test_tensor_cnn = torch.tensor(y_test1, dtype=torch.long)

# 画像データを[0, 1]の範囲に正規化 (MNISTデータは通常0-255)
X_train_tensor_cnn /= 255.0
X_test_tensor_cnn /= 255.0

# DatasetとDataLoaderの作成
train_dataset_cnn = TensorDataset(X_train_tensor_cnn, y_train_tensor_cnn)
test_dataset_cnn = TensorDataset(X_test_tensor_cnn, y_test_tensor_cnn)

batch_size = 64
train_loader_cnn = DataLoader(train_dataset_cnn, batch_size=batch_size, shuffle=True)
test_loader_cnn = DataLoader(test_dataset_cnn, batch_size=batch_size, shuffle=False)

# より深いCNNモデルの定義
class Deeper_MNIST_CNN(nn.Module):
    def __init__(self):
        super(Deeper_MNIST_CNN, self).__init__()
        # 畳み込み層1: 入力1チャンネル、出力32チャンネル、カーネルサイズ3x3、パディング1
        # 入力: (batch_size, 1, 28, 28) -> 出力: (batch_size, 32, 28, 28)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        # プーリング層1: カーネルサイズ2x2、ストライド2
        # 入力: (batch_size, 32, 28, 28) -> 出力: (batch_size, 32, 14, 14)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # 畳み込み層2: 入力32チャンネル、出力64チャンネル、カーネルサイズ3x3、パディング1
        # 入力: (batch_size, 32, 14, 14) -> 出力: (batch_size, 64, 14, 14)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        # プーリング層2: カーネルサイズ2x2、ストライド2
        # 入力: (batch_size, 64, 14, 14) -> 出力: (batch_size, 64, 7, 7)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # 新しい畳み込み層3: 入力64チャンネル、出力128チャンネル、カーネルサイズ3x3、パディング1
        # 入力: (batch_size, 64, 7, 7) -> 出力: (batch_size, 128, 7, 7)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        # 新しいプーリング層3: カーネルサイズ2x2、ストライド2
        # 入力: (batch_size, 128, 7, 7) -> 出力: (batch_size, 128, 3, 3) (7/2=3.5なので切り捨て)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # 全結合層: 最後のプーリング層の出力サイズ (128 * 3 * 3) から10クラスへ
        self.fc = nn.Linear(128 * 3 * 3, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = x.view(-1, 128 * 3 * 3) # フラット化
        x = self.fc(x)
        return x

# モデルのインスタンス化、損失関数、オプティマイザの定義
model_deeper_cnn = Deeper_MNIST_CNN()
criterion_deeper_cnn = nn.CrossEntropyLoss()
optimizer_deeper_cnn = optim.Adam(model_deeper_cnn.parameters(), lr=0.001)

# デバイスの設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_deeper_cnn.to(device)

# 訓練中のメトリクスを保存するリスト
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

# モデルの訓練
epochs = 10
print("\nより深いCNNモデル訓練開始...")
for epoch in range(epochs):
    model_deeper_cnn.train()
    running_loss = 0.0
    correct_train_epoch = 0
    total_train_epoch = 0
    # tqdm で train_loader_cnn をラップしてプログレスバーを表示
    train_loader_tqdm = tqdm(train_loader_cnn, desc=f"Epoch {epoch+1}/{epochs} (Train)")
    for inputs, labels in train_loader_tqdm:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_deeper_cnn.zero_grad()
        outputs = model_deeper_cnn(inputs)
        loss = criterion_deeper_cnn(outputs, labels)
        loss.backward()
        optimizer_deeper_cnn.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train_epoch += labels.size(0)
        correct_train_epoch += (predicted == labels).sum().item()
        # プログレスバーに現在の損失と精度を表示
        train_loader_tqdm.set_postfix(loss=running_loss/len(train_loader_tqdm), acc=100.*correct_train_epoch/total_train_epoch)

    train_loss_epoch = running_loss / len(train_loader_cnn)
    train_accuracy_epoch = 100 * correct_train_epoch / total_train_epoch
    train_losses.append(train_loss_epoch)
    train_accuracies.append(train_accuracy_epoch)

    # エポック終了後のテストデータでの評価
    model_deeper_cnn.eval()
    correct_test_epoch = 0
    total_test_epoch = 0
    running_test_loss = 0.0
    # tqdm で test_loader_cnn をラップしてプログレスバーを表示
    test_loader_tqdm = tqdm(test_loader_cnn, desc=f"Epoch {epoch+1}/{epochs} (Test)")
    with torch.no_grad():
        for inputs, labels in test_loader_tqdm: # テストデータでの精度
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_deeper_cnn(inputs)
            loss = criterion_deeper_cnn(outputs, labels)
            running_test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_test_epoch += labels.size(0)
            correct_test_epoch += (predicted == labels).sum().item()
            # プログレスバーに現在の損失と精度を表示
            test_loader_tqdm.set_postfix(loss=running_test_loss/len(test_loader_tqdm), acc=100.*correct_test_epoch/total_test_epoch)

    test_loss_epoch = running_test_loss / len(test_loader_cnn)
    test_accuracy_epoch = 100 * correct_test_epoch / total_test_epoch

    test_accuracies.append(test_accuracy_epoch)
    test_losses.append(test_loss_epoch)

    print(f"Epoch {epoch+1}, Train Loss: {train_loss_epoch:.4f}, Train Acc: {train_accuracy_epoch:.2f}%, Test Loss: {test_loss_epoch:.4f}, Test Acc: {test_accuracy_epoch:.2f}%")

print("より深いCNNモデル訓練完了！")

# モデルの評価 (最終エポックの結果)
model_deeper_cnn.eval()
correct_deeper_cnn = 0
total_deeper_cnn = 0
all_predicted_deeper_cnn = []
all_labels_deeper_cnn = []
with torch.no_grad():
    for inputs, labels in test_loader_cnn:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model_deeper_cnn(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total_deeper_cnn += labels.size(0)
        correct_deeper_cnn += (predicted == labels).sum().item()
        all_predicted_deeper_cnn.extend(predicted.cpu().numpy())
        all_labels_deeper_cnn.extend(labels.cpu().numpy())

accuracy_deeper_cnn = 100 * correct_deeper_cnn / total_deeper_cnn
print(f"\n最終テストデータでのより深いCNNモデル精度: {accuracy_deeper_cnn:.2f}%")

# 詳細なレポート
print("\nより深いCNNモデルの分類レポート:")
print(classification_report(all_labels_deeper_cnn, all_predicted_deeper_cnn))

print("\nより深いCNNモデルの混同行列:")
print(confusion_matrix(all_labels_deeper_cnn, all_predicted_deeper_cnn))

# 学習曲線 (Learning Curves) のプロット
plt.figure(figsize=(12, 5))

# 損失のプロット
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), train_losses, label='Train Loss')
plt.plot(range(1, epochs + 1), test_losses, label='Test Loss')
plt.title('Loss per Epoch (Deeper CNN)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 精度のプロット
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), train_accuracies, label='Train Accuracy')
plt.plot(range(1, epochs + 1), test_accuracies, label='Test Accuracy')
plt.title('Accuracy per Epoch (Deeper CNN)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### `Deeper_MNIST_CNN` モデルの構造

In [ ]:
print(model_deeper_cnn)

In [ ]:
import pandas as pd

def get_model_summary(model):
    summary = []
    for name, module in model.named_children():
        layer_info = {
            'Layer Name': name,
            'Layer Type': str(type(module)).split('.')[-1].replace("'>", ""),
            'Details': ''
        }

        if isinstance(module, nn.Conv2d):
            layer_info['Details'] = (
                f'In Channels: {module.in_channels}, '
                f'Out Channels: {module.out_channels}, '
                f'Kernel Size: {module.kernel_size}, '
                f'Stride: {module.stride}, '
                f'Padding: {module.padding}'
            )
        elif isinstance(module, nn.MaxPool2d):
            layer_info['Details'] = (
                f'Kernel Size: {module.kernel_size}, '
                f'Stride: {module.stride}'
            )
        elif isinstance(module, nn.Linear):
            layer_info['Details'] = (
                f'In Features: {module.in_features}, '
                f'Out Features: {module.out_features}'
            )
        elif isinstance(module, nn.ReLU):
            layer_info['Details'] = 'In-place: False'

        summary.append(layer_info)
    return pd.DataFrame(summary)

model_summary_df = get_model_summary(model_deeper_cnn)
display(model_summary_df)

aa